## Function/Tool Calling for Airline Booking

In [1]:
import ollama
# import json
import pprint


In [2]:
# Function for getting the prices of a Flight
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}
MODEL = "mistral:7b"
def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown Ticket Price")
    return price
    # return f"The price of a ticket to {destination_city} is {price}"

In [3]:
# Below is th dictionary structure needed to describe our function in order for the llm to match the user query to our function
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer want to travel to"
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [4]:
# During the llm call we need to pass the messages with the concatenated history and also another
# essential field called the tools
# this is a essential feature that modern frontier models support even the open source one.

#You can see below we pass the function description and not the actual function
tools = [get_ticket_price]

In [5]:
system_prompt = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
Ask for destination city and call the necessary tool for the same"""

In [6]:
# def chat(message, history):
#     history = [{"role": h["role"], "content":h["content"]} for h in history]
#     messages = [{"role":"system","content":system_prompt}] + history + [{"role":"user","content":message}]
#     response = ollama.chat(model=MODEL, messages=messages, tools=tools)
#     pprint.pprint(response.model_dump_json())

#     responses = []
#     # Now checking the finish_reason parameter of the llm's output and not the message
#     # if the finish_reason parameter is set to "tool_calls" specifically then we check which function it is calling to 
#     # in our case it is get_ticket_price using the price_function description.
#     if response.message.tool_calls:
#         for call in response.message.tool_calls:
#             print(call)
#             message = response.message.content
#             messages.append({
#                 "role":"assistant",
#                 "content":message
#                 })

#             print(call.function.name)
#             print(call.function.arguments)
#             if call.function.name == "get_ticket_price":
#                 arguments = call.function.arguments
#                 city = arguments.get('destination_city')
#                 price_details = get_ticket_price(city)
#                 responses.append({
#                     "role":"tool",
#                     "content": price_details,
#                     # "tool_call_id": call.id
#                 })
#             messages.extend(responses)
#             response = ollama.chat(model=MODEL, messages=messages)
#     return response.message.content            

In [7]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticekt_price":
            arguments = tool_call.function.arguments
            city = arguments.get("destination_city")
            price_details = get_ticket_price(city)
            responses.append({
                "role":"tool",
                "content": price_details
            })
    return responses

In [ ]:
def chat(message,history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system", "content":system_prompt}] + history + [{"role":"user","content":message}]
    response = ollama.chat(model=MODEL,messages=messages,tools=tools)

    if response.message.tool_calls:
        message = response.content.message
        responses = handle_tool_calls(message)
        messages.append({
            "role": "assistant",
            "content": message
        })
        messages.extend(responses)
        response = ollama.chat(model=MODEL, messages = messages)
    return response.content.message

In [9]:
import gradio as gr

gr.ChatInterface(fn=chat,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/skrrr/Projects/llm_engineering/llms/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/skrrr/Projects/llm_engineering/llms/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/skrrr/Projects/llm_engineering/llms/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/skrrr/Projects/llm_engineering/llms/lib/python3.12/site-packages/gradio/blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/skrrr/Projects/llm_engineering/llms/lib/python3.12/site-packages/gradio/ut